In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import paths
from homography import HomographyConfig

hom_cfg = HomographyConfig()

# ---- load the two frame tables built in match_frame_table.ipynb ----
player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)

print("player_frame_table:", player_frame_table.shape)
print("ball_frame_table:  ", ball_frame_table.shape)
print("columns (player):", list(player_frame_table.columns))
print("columns (ball):  ", list(ball_frame_table.columns))

FPS = 25  # confirm against your source video

# ---- on-ball press config (individual carrier pressure) ----
PRESS_DEFENDER_DIST_M = 3.0
PRESS_RADIUS_M = 5.0
PRESS_MIN_DEFENDERS_IN_RADIUS = 2
PRESS_BALL_SPEED_THRESH_MPS = 3.0
PRESS_MIN_CONSECUTIVE_FRAMES = 5

# ---- team press config (collective, per possession phase) ----
TEAM_PRESS_LINE_HEIGHT_THRESH_M = 35.0   # defending team's backline must be at least this far up the pitch (from own goal) to count as "pushed up" -- tune after inspection
TEAM_PRESS_MIN_CONVERGING_DEFENDERS = 3  # how many defenders must be closing distance toward the ball for numbers-based convergence
TEAM_PRESS_CONVERGE_RADIUS_M = 15.0      # radius around the ball within which a defender's movement counts toward convergence

# ---- team label check ----
print("\nUnique team values in player_frame_table:", sorted(player_frame_table["team"].dropna().unique().tolist()))

c:\Users\user\Desktop\Football_cv_project\football_pressure_analysis\venv_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


player_frame_table: (70396, 10)
ball_frame_table:   (3001, 10)
columns (player): ['frame_idx', 'track_id', 'role', 'team', 'x_px', 'y_px', 'pitch_x', 'pitch_y', 'is_carrier', 'attack_direction']
columns (ball):   ['frame_idx', 'ball_x_px', 'ball_y_px', 'ball_pitch_x', 'ball_pitch_y', 'ball_source', 'ball_conf', 'carrier_track_id', 'carrier_team', 'possession_team']

Unique team values in player_frame_table: [0, 1]


In [2]:
def build_possession_phases(ball_frame_table):
    df = ball_frame_table.sort_values("frame_idx").reset_index(drop=True)
    valid = df["possession_team"].notna()
    df = df[valid].copy()

    runs = (df["possession_team"] != df["possession_team"].shift()).cumsum()
    phases = (
        df.groupby(runs)
        .agg(
            start_frame=("frame_idx", "min"),
            end_frame=("frame_idx", "max"),
            possession_team=("possession_team", "first"),
        )
        .reset_index(drop=True)
    )
    phases["duration_frames"] = phases["end_frame"] - phases["start_frame"] + 1
    phases["defending_team"] = phases["possession_team"].apply(lambda t: 1 - int(t))
    return phases


possession_phases = build_possession_phases(ball_frame_table)

print(f"Total possession phases: {len(possession_phases)}")
print(f"Phase duration (frames) -> mean: {possession_phases['duration_frames'].mean():.1f}, "
      f"min: {possession_phases['duration_frames'].min()}, max: {possession_phases['duration_frames'].max()}")
print(f"Phase duration (sec)    -> mean: {possession_phases['duration_frames'].mean()/FPS:.2f}s, "
      f"max: {possession_phases['duration_frames'].max()/FPS:.2f}s")

possession_phases.head(10)

Total possession phases: 11
Phase duration (frames) -> mean: 272.2, min: 10, max: 1386
Phase duration (sec)    -> mean: 10.89s, max: 55.44s


,start_frame,end_frame,possession_team,duration_frames,defending_team
0,7,279,1,273,0
1,280,1665,0,1386,1
2,1666,1842,1,177,0
3,1843,1852,0,10,1
4,1853,2118,1,266,0
5,2119,2133,0,15,1
6,2134,2161,1,28,0
7,2162,2737,0,576,1
8,2738,2827,1,90,0
9,2828,2841,0,14,1


In [5]:
player_by_frame = {
    f: g[["track_id", "team", "role", "pitch_x", "pitch_y", "is_carrier", "attack_direction"]].reset_index(drop=True)
    for f, g in player_frame_table.groupby("frame_idx")
}
print("player_by_frame frames:", len(player_by_frame))


def defending_line_height(frame_df, defending_team, pitch_length):
    """Returns how far up the pitch the defending team's backline is --
    i.e. the deepest outfield player's distance from their own goal line.
    Lower number = deeper/more defensive; higher number = pushed up higher
    (pressing posture). Excludes goalkeeper. Uses attack_direction so this
    is meaningful regardless of which physical end the team defends."""
    team_df = frame_df[
        (frame_df["team"] == defending_team)
        & (frame_df["role"] == "player")
        & frame_df["pitch_x"].notna()
    ]
    if team_df.empty:
        return np.nan

    direction = team_df["attack_direction"].dropna()
    if direction.empty:
        return np.nan
    attack_dir = direction.iloc[0]

    if attack_dir == 1:
        return team_df["pitch_x"].min()
    else:
        return pitch_length - team_df["pitch_x"].max()


test_frames = [10, 50, 100, 200]
for f in test_frames:
    fdf = player_by_frame.get(f)
    height = defending_line_height(fdf, defending_team=0, pitch_length=hom_cfg.pitch_length)
    print(f"frame {f}: defending_team=0 line_height={height}")

player_by_frame frames: 3001
frame 10: defending_team=0 line_height=31.852548599243164
frame 50: defending_team=0 line_height=33.75464630126953
frame 100: defending_team=0 line_height=37.25908660888672
frame 200: defending_team=0 line_height=34.08921432495117


In [6]:
def defenders_converging(frame_idx, prev_frame_idx, player_by_frame, ball_by_frame,
                          defending_team, radius_m=TEAM_PRESS_CONVERGE_RADIUS_M):
    """Counts defenders within radius_m of the ball this frame whose
    distance to the ball decreased vs the previous frame -- i.e. actively
    closing down, not just standing nearby. Returns (n_converging, n_in_radius)."""
    curr_df = player_by_frame.get(frame_idx)
    prev_df = player_by_frame.get(prev_frame_idx)
    if curr_df is None or prev_df is None:
        return np.nan, np.nan
    if frame_idx not in ball_by_frame.index or prev_frame_idx not in ball_by_frame.index:
        return np.nan, np.nan

    ball_curr = ball_by_frame.loc[frame_idx]
    ball_prev = ball_by_frame.loc[prev_frame_idx]
    if pd.isna(ball_curr["ball_pitch_x"]) or pd.isna(ball_prev["ball_pitch_x"]):
        return np.nan, np.nan

    defenders_curr = curr_df[(curr_df["team"] == defending_team) & curr_df["pitch_x"].notna()]
    defenders_prev = prev_df[(prev_df["team"] == defending_team) & prev_df["pitch_x"].notna()].set_index("track_id")

    n_in_radius = 0
    n_converging = 0
    for _, row in defenders_curr.iterrows():
        dist_curr = np.hypot(row["pitch_x"] - ball_curr["ball_pitch_x"], row["pitch_y"] - ball_curr["ball_pitch_y"])
        if dist_curr > radius_m:
            continue
        n_in_radius += 1
        if row["track_id"] in defenders_prev.index:
            prow = defenders_prev.loc[row["track_id"]]
            dist_prev = np.hypot(prow["pitch_x"] - ball_prev["ball_pitch_x"], prow["pitch_y"] - ball_prev["ball_pitch_y"])
            if dist_curr < dist_prev:
                n_converging += 1

    return n_converging, n_in_radius


ball_by_frame = ball_frame_table.set_index("frame_idx")

for f in [101, 150, 201, 300]:
    n_conv, n_rad = defenders_converging(f, f - 1, player_by_frame, ball_by_frame, defending_team=0)
    print(f"frame {f}: converging={n_conv}, in_radius={n_rad}")

frame 101: converging=2, in_radius=2
frame 150: converging=2, in_radius=2
frame 201: converging=0, in_radius=2
frame 300: converging=1, in_radius=7


In [7]:
def frame_to_defending_team(frame_idx, possession_phases):
    """Look up defending_team for a given frame from possession_phases.
    Returns None if the frame isn't covered by any phase (e.g. before the
    first possession is ever established)."""
    match = possession_phases[
        (possession_phases["start_frame"] <= frame_idx) & (possession_phases["end_frame"] >= frame_idx)
    ]
    if match.empty:
        return None
    return int(match.iloc[0]["defending_team"])


def compute_team_press_trigger(frame_idx, player_by_frame, ball_by_frame, possession_phases,
                                pitch_length,
                                line_height_thresh=TEAM_PRESS_LINE_HEIGHT_THRESH_M,
                                min_converging=TEAM_PRESS_MIN_CONVERGING_DEFENDERS,
                                converge_radius=TEAM_PRESS_CONVERGE_RADIUS_M):
    defending_team = frame_to_defending_team(frame_idx, possession_phases)
    if defending_team is None:
        return False, {"reason": "no_phase"}

    frame_df = player_by_frame.get(frame_idx)
    if frame_df is None:
        return False, {"reason": "no_frame_data"}

    line_height = defending_line_height(frame_df, defending_team, pitch_length)
    n_converging, n_in_radius = defenders_converging(
        frame_idx, frame_idx - 1, player_by_frame, ball_by_frame, defending_team, converge_radius
    )

    line_pushed_up = (not np.isnan(line_height)) and line_height >= line_height_thresh
    numbers_converging = (not np.isnan(n_converging)) and n_converging >= min_converging

    triggered = line_pushed_up and numbers_converging

    info = {
        "defending_team": defending_team,
        "line_height_m": line_height,
        "n_converging": n_converging,
        "n_in_radius": n_in_radius,
        "line_pushed_up": line_pushed_up,
        "numbers_converging": numbers_converging,
    }
    return triggered, info


# smoke test across the same frames used before
for f in [101, 150, 201, 300]:
    trig, info = compute_team_press_trigger(f, player_by_frame, ball_by_frame, possession_phases, hom_cfg.pitch_length)
    print(f"frame {f}: team_press_trigger={trig}  {info}")

frame 101: team_press_trigger=False  {'defending_team': 0, 'line_height_m': 32.9996452331543, 'n_converging': 2, 'n_in_radius': 2, 'line_pushed_up': False, 'numbers_converging': False}
frame 150: team_press_trigger=False  {'defending_team': 0, 'line_height_m': 36.90339279174805, 'n_converging': 2, 'n_in_radius': 2, 'line_pushed_up': True, 'numbers_converging': False}
frame 201: team_press_trigger=False  {'defending_team': 0, 'line_height_m': 33.91861343383789, 'n_converging': 0, 'n_in_radius': 2, 'line_pushed_up': False, 'numbers_converging': False}
frame 300: team_press_trigger=False  {'defending_team': 1, 'line_height_m': 40.65782928466797, 'n_converging': 1, 'n_in_radius': 5, 'line_pushed_up': True, 'numbers_converging': False}


In [8]:
records = []
for f in range(int(ball_frame_table["frame_idx"].max()) + 1):
    trig, info = compute_team_press_trigger(f, player_by_frame, ball_by_frame, possession_phases, hom_cfg.pitch_length)
    info["frame_idx"] = f
    info["team_press_trigger_raw"] = trig
    records.append(info)

team_press_df = pd.DataFrame(records)

print("Total frames:", len(team_press_df))
print("\nReason breakdown for skipped frames:")
print(team_press_df["reason"].value_counts(dropna=False) if "reason" in team_press_df.columns else "n/a")

print("\nRaw team_press_trigger_raw rate:", team_press_df["team_press_trigger_raw"].mean())
print("Raw trigger count:", team_press_df["team_press_trigger_raw"].sum())

print("\nline_height_m distribution:")
print(team_press_df["line_height_m"].describe())

print("\nn_converging distribution:")
print(team_press_df["n_converging"].describe())

print("\nline_pushed_up rate:", team_press_df["line_pushed_up"].mean())
print("numbers_converging rate:", team_press_df["numbers_converging"].mean())

Total frames: 3001

Reason breakdown for skipped frames:
reason
NaN         2994
no_phase       7
Name: count, dtype: int64

Raw team_press_trigger_raw rate: 0.08363878707097634
Raw trigger count: 251

line_height_m distribution:
count    2994.000000
mean       39.894769
std        10.872903
min        12.572067
25%        30.827663
50%        41.389145
75%        49.682608
max        61.288208
Name: line_height_m, dtype: float64

n_converging distribution:
count    2703.000000
mean        1.442471
std         1.286611
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         7.000000
Name: n_converging, dtype: float64

line_pushed_up rate: 0.6018704074816299
numbers_converging rate: 0.1646626586506346
